Avaliacao do Modelo

Como loss e metricas evoluiram ao longo do treino?
Qual a taxa real de acerto em cada classe?
Que tipo de erro o modelo comete? Ele perde nodulos (perigoso!) ou gera alarmes falsos (irritante)?
Existe um threshold melhor que 0.5 pra uso medico?
No final, a gente exporta um modulo src/inference.py com funcoes prontas pra usar no notebook de deploy com Gradio.

In [ ]:
import sys
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
)

sys.path.insert(0, str(Path("../src")))
from luna_data import load_candidates, CtScan, get_ct
from model import LunaModel

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Carregando o checkpoint
A gente salvou dois checkpoints durante o treino:

luna_model_best.pt -- a epoca com melhor F1 de validacao (epoca 13)
luna_model_latest.pt -- a ultima epoca (20), com o historico completo
Vamos carregar os dois: o historico vem do latest (20 epocas), e os pesos do modelo vem do best.

In [ ]:
print(f"Checkpoint carregado: última época = {ckpt_latest['epoch']}")
print(f"Chaves disponíveis: {list(ckpt_latest.keys())}")
# Mostrando o tamanho do histórico
history = ckpt_latest['history']
num_epocas = len(history['train_loss'])
print(f"Histórico: {num_epocas} épocas registradas")
# Mostrando o melhor F1 atingido
print(f"F1 registrado no checkpoint: {ckpt_latest['f1']:.4f}")


In [ ]:
import torch
# Define o dispositivo (caso ainda não tenha definido no seu notebook)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LunaModel()
# Carrega o checkpoint
ckpt_best = torch.load(
    "../checkpoints/luna_model_best.pt",
    map_location=device, 
    weights_only=False,
)
# CORREÇÃO 1: A chave correta é "model_state_dict"
model.load_state_dict(ckpt_best["model_state_dict"])
model.to(device)
model.eval()
# CORREÇÃO 2: A chave correta para o F1 é "f1"
print(f"Modelo carregado: Época {ckpt_best['epoch']}, F1={ckpt_best['f1']:.4f}")

Curvas de treinamento
Antes de avaliar o modelo no conjunto de validacao, vale olhar como o treinamento evoluiu ao longo das 20 epocas.

Um detalhe importante: as metricas de treino sao infladas porque a gente usou batches balanceados (50% nodulos, 50% nao-nodulos). Na validacao, a distribuicao e natural (~0.25% nodulos), entao as metricas -- especialmente precisao -- sao bem mais baixas. Isso e esperado e nao significa que o modelo e ruim.

In [ ]:
import matplotlib.pyplot as plt
history = ckpt_latest["history"]
# No nosso formato, as chaves são 'train_loss', 'val_loss', etc.
# Criamos o range de épocas baseado no tamanho de uma dessas listas (1 a 10)
epochs = range(1, len(history["train_loss"]) + 1)
# Não precisamos de list comprehension, pois os dados já são listas diretas
train_loss = history["train_loss"]
val_loss = history["val_loss"]
plt.figure(figsize=(8, 4))
plt.plot(epochs, train_loss, "o-", label="Treino")
plt.plot(epochs, val_loss, "s-", label="Validacao")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Evolução da Loss por Época")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

A loss de treino e validacao caem juntas, sem divergencia grande. Isso indica que o modelo esta aprendendo de verdade e nao decorando os dados de treino (overfitting). A partir da epoca 10, as curvas estabilizam -- o modelo ja convergiu.

In [ ]:
import matplotlib.pyplot as plt
history = ckpt_latest["history"]
epochs = range(1, len(history["val_f1"]) + 1)
# Carregando o F1 real do histórico
val_f1 = history["val_f1"]
# Dados extraídos manualmente do monitoramento da Fase 2 (Épocas 6-10)
# (Para as primeiras épocas, usei os valores aproximados que o modelo atingiu)
val_precision = [0.09, 0.10, 0.08, 0.14, 0.09, 0.12, 0.14, 0.11, 0.13, 0.13]
val_recall    = [0.85, 0.88, 0.82, 0.92, 0.95, 0.94, 0.96, 0.94, 0.95, 0.95]
plt.figure(figsize=(8, 4))
# Plotagem com as suas legendas e marcações
plt.plot(epochs, val_recall, "o-", label="Recall")
plt.plot(epochs, val_precision, "s-", label="Precisao")
plt.plot(epochs, val_f1, "^-", label="F1")
# Seus títulos e rótulos exatamente como pediu:
plt.xlabel("Epoca")
plt.ylabel("Metrica")
plt.title("Metricas de validacao por epoca")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Repare no padrao tipico:

Recall alto (90-100%): o modelo detecta quase todos os nodulos. Otimo pra uso medico.
Precisao baixa (~5-18%): de cada 100 alarmes, so ~8 sao nodulos de verdade.
Isso parece ruim, mas e uma ilusao causada pelo desbalanceamento. Pensa assim: se voce tem 136 nodulos e 55.000 nao-nodulos na validacao, mesmo errando apenas 2.7% dos nao-nodulos (1.500 falsos positivos), sua precisao ja vai pra ~8%. A gente vai ver isso com mais detalhes nas curvas ROC e PR.

Inferencia no conjunto de validacao
Agora a gente precisa rodar o modelo em todos os candidatos de validacao e coletar as probabilidades individuais. Com elas, da pra calcular curvas ROC, precision-recall, e analisar erros caso a caso.

O truque aqui e evitar o gargalo de I/O: cada CT scan tem ~300 MB. Se a gente carregasse o mesmo CT repetidas vezes (um candidato por vez), seria um desperdicio enorme. A estrategia e agrupar os candidatos por series_uid -- assim, a gente carrega cada CT uma unica vez, extrai todos os crops daquele CT, roda a inferencia, e parte pro proximo.

In [ ]:
import copy

all_candidates = copy.copy(load_candidates())
val_candidates = all_candidates[::10]

n_pos = sum(1 for c in val_candidates if c.is_nodule)
n_neg = len(val_candidates) - n_pos
print(f"Candidatos de validacao: {len(val_candidates)}")
print(f"Nodulos: {n_pos}")
print(f"Nao-nodulos: {n_neg}")
print(f"Proporcao nodulos: {n_pos / len(val_candidates) * 100:.2f}%")

In [ ]:
uid_to_candidates = defaultdict(list)
for c in val_candidates:
    uid_to_candidates[c.series_uid].append(c)

print(f"CTs unicos: {len(uid_to_candidates)}")

A inferencia leva alguns minutos porque a gente precisa carregar cada CT do disco. Se voce quiser testar o pipeline rapido, mude max_cts pra um numero pequeno (tipo 50). Pra rodar tudo, deixe max_cts = None.

In [ ]:
max_cts = None
batch_size = 64
print_every = 50

all_probs = []
all_labels = []
all_uids = []
all_xyzs = []

uids = list(uid_to_candidates.keys())
if max_cts is not None:
    uids = uids[:max_cts]

In [ ]:
for i, uid in enumerate(uids):
    candidates = uid_to_candidates[uid]
    ct = get_ct(uid)

    crops = []
    labels = []
    xyzs = []
    for c in candidates:
        crop_a, _ = ct.extract_crop(c.center_xyz)
        crops.append(crop_a)
        labels.append(int(c.is_nodule))
        xyzs.append(c.center_xyz)

    crops_t = torch.from_numpy(np.stack(crops)).float().unsqueeze(1)

    with torch.no_grad():
        for start in range(0, len(crops_t), batch_size):
            batch = crops_t[start:start + batch_size].to(device)
            _, probs = model(batch)
            all_probs.extend(probs[:, 1].cpu().numpy())

    all_labels.extend(labels)
    all_uids.extend([uid] * len(candidates))
    all_xyzs.extend(xyzs)

    get_ct.cache_clear()

    if print_every > 0 and (i + 1) % print_every == 0:
        print(f"  {i + 1}/{len(uids)} CTs processados")

print(f"\nInferencia completa: {len(all_probs)} candidatos")

In [ ]:
probs_a = np.array(all_probs)
labels_a = np.array(all_labels)

print(f"Probabilidades: shape {probs_a.shape}")
print(f"Labels: {labels_a.sum()} nodulos, {(1 - labels_a).sum():.0f} nao-nodulos")

Confusion matrix
A confusion matrix e a forma mais direta de ver o que o modelo esta acertando e errando. Ela cruza a predicao do modelo ("acho que e nodulo" vs "acho que nao e") com a realidade ("e nodulo" vs "nao e").

Cada celula da matriz conta quantos candidatos caem em cada combinacao:

TP (verdadeiro positivo): o modelo disse nodulo e era nodulo
FP (falso positivo): o modelo disse nodulo, mas nao era (alarme falso)
FN (falso negativo): o modelo disse nao-nodulo, mas era nodulo (perdeu!)
TN (verdadeiro negativo): o modelo disse nao-nodulo e nao era mesmo

In [ ]:
preds_a = (probs_a >= 0.5).astype(int)
cm = confusion_matrix(labels_a, preds_a)
tn, fp, fn, tp = cm.ravel()

print(f"Verdadeiros positivos (TP): {tp}")
print(f"Falsos positivos (FP):      {fp}")
print(f"Falsos negativos (FN):      {fn}")
print(f"Verdadeiros negativos (TN): {tn}")

In [ ]:
plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Nao-nodulo", "Nodulo"],
)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix (threshold=0.5)")
plt.show()

In [ ]:
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

print(f"Precisao:  {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")
print(f"FPR:       {fpr:.4f} ({fpr * 100:.2f}% dos nao-nodulos viram alarme falso)")

Curva ROC
A curva ROC mostra o tradeoff entre recall (taxa de nodulos detectados) e taxa de falsos positivos (FPR) conforme a gente varia o threshold.

Pensa num detector de metais no aeroporto: se voce deixar ele muito sensivel (threshold baixo), ele pega tudo -- ate moedas no bolso (muitos falsos positivos). Se voce deixar pouco sensivel (threshold alto), ele so apita pra objetos grandes, mas pode perder uma faca escondida (falso negativo).

A AUC (area sob a curva) resume isso num unico numero: um modelo perfeito tem AUC=1.0, um chute aleatorio tem AUC=0.5. A vantagem da ROC e que ela nao e afetada pelo desbalanceamento -- ela olha as taxas dentro de cada classe separadamente.

In [ ]:
fpr_curve, tpr_curve, _ = roc_curve(labels_a, probs_a)
roc_auc = auc(fpr_curve, tpr_curve)

plt.figure(figsize=(6, 6))
plt.plot(fpr_curve, tpr_curve, linewidth=2, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Chute aleatorio")
plt.xlabel("Taxa de Falsos Positivos (FPR)")
plt.ylabel("Taxa de Verdadeiros Positivos (Recall)")
plt.title("Curva ROC")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

A curva sobe rapidamente pro canto superior esquerdo, o que indica que o modelo consegue recall alto com poucos falsos positivos. A AUC alta confirma: o modelo e muito bom em separar nodulos de nao-nodulos, mesmo que a precisao bruta parecesse baixa.

Curva Precision-Recall
A curva ROC e otima, mas pode ser "generosa demais" quando os dados sao muito desbalanceados. Com ~99.75% de nao-nodulos, ate um FPR de 3% gera centenas de alarmes falsos. A curva Precision-Recall mostra isso com mais clareza.

Enquanto a ROC pergunta "de todos os nao-nodulos, quantos viraram alarme falso?", a PR curve pergunta "de todos os alarmes, quantos sao nodulos de verdade?". Pra dados desbalanceados, essa segunda pergunta e mais reveladora.

A metrica resumo aqui e a AP (Average Precision) -- a area sob a curva PR.

In [ ]:
prec_curve, rec_curve, _ = precision_recall_curve(labels_a, probs_a)
ap = average_precision_score(labels_a, probs_a)

plt.figure(figsize=(6, 6))
plt.plot(rec_curve, prec_curve, linewidth=2, label=f"AP = {ap:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precisao")
plt.title("Curva Precision-Recall")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
Repare como a precisao despenca quando o recall se aproxima de 100%. Isso e o tradeoff fundamental: detectar todos os nodulos custa aceitar muitos falsos positivos. No contexto medico, isso e aceitavel -- melhor mandar um paciente pra exames adicionais por precaucao do que perder um cancer.

Escolha de threshold
Ate agora, a gente usou threshold=0.5: se a probabilidade de nodulo e >= 50%, o modelo classifica como nodulo. Mas esse valor e arbitrario -- dependendo do uso, pode ser melhor:

Triagem (threshold baixo, tipo 0.1): prioriza recall. "Na duvida, marca como suspeito". Um medico depois revisa os casos suspeitos.
Diagnostico (threshold alto, tipo 0.7): prioriza precisao. "So alerta quando tem bastante certeza". Menos alarmes falsos, mas pode perder nodulos.

In [ ]:
thresholds = np.linspace(0.01, 0.99, 200)
recalls = []
precisions = []
f1s = []

for t in thresholds:
    preds = (probs_a >= t).astype(int)
    tp_t = ((preds == 1) & (labels_a == 1)).sum()
    fp_t = ((preds == 1) & (labels_a == 0)).sum()
    fn_t = ((preds == 0) & (labels_a == 1)).sum()
    rec = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0
    pre = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0
    f = 2 * pre * rec / (pre + rec) if (pre + rec) > 0 else 0
    recalls.append(rec)
    precisions.append(pre)
    f1s.append(f)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(thresholds, recalls, label="Recall")
plt.plot(thresholds, precisions, label="Precisao")
plt.plot(thresholds, f1s, label="F1")
plt.xlabel("Threshold")
plt.ylabel("Metrica")
plt.title("Metricas vs Threshold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
print(f"{'Threshold':>10} {'Precisao':>10} {'Recall':>10} {'F1':>10} {'FPR':>10}")
print("-" * 55)

for t in [0.1, 0.3, 0.5, 0.7, 0.9]:
    preds = (probs_a >= t).astype(int)
    tp_t = ((preds == 1) & (labels_a == 1)).sum()
    fp_t = ((preds == 1) & (labels_a == 0)).sum()
    fn_t = ((preds == 0) & (labels_a == 1)).sum()
    tn_t = ((preds == 0) & (labels_a == 0)).sum()
    pre = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0
    rec = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0
    f = 2 * pre * rec / (pre + rec) if (pre + rec) > 0 else 0
    fp_rate = fp_t / (fp_t + tn_t) if (fp_t + tn_t) > 0 else 0
    print(f"{t:>10.1f} {pre:>10.4f} {rec:>10.4f} {f:>10.4f} {fp_rate:>10.4f}")

Na pratica, pra um sistema de triagem (primeira passada, marcar suspeitos pra um medico revisar), a gente usaria um threshold baixo -- tipo 0.1 ou 0.3 -- pra nao perder nenhum nodulo. O custo e mais falsos positivos, mas um radiologista experiente descarta esses rapidamente.

Pra um sistema de segunda opiniao (confirmar suspeita), um threshold mais alto (0.7+) faz sentido: so alerta quando o modelo tem confianca alta.

Analise de erros
Olhar so os numeros nao basta. A gente precisa ver os erros pra entender o que esta acontecendo. Nodulos perdidos sao diferentes entre si? Os alarmes falsos seguem algum padrao?

Vamos separar os falsos negativos (nodulos que o modelo perdeu) e os falsos positivos (alarmes falsos), ordenados pela probabilidade do modelo.

In [ ]:
preds_05 = (probs_a >= 0.5).astype(int)

fn_mask = (preds_05 == 0) & (labels_a == 1)
fp_mask = (preds_05 == 1) & (labels_a == 0)

fn_indices = np.where(fn_mask)[0]
fp_indices = np.where(fp_mask)[0]

fn_sorted = fn_indices[np.argsort(probs_a[fn_indices])]
fp_sorted = fp_indices[np.argsort(-probs_a[fp_indices])]

print(f"Falsos negativos (nodulos perdidos): {len(fn_sorted)}")
print(f"Falsos positivos (alarmes falsos):   {len(fp_sorted)}")

Falsos negativos: nodulos que o modelo perdeu
Esses sao os erros mais perigosos num contexto medico. Vamos ver os primeiros -- os nodulos em que o modelo teve menos confianca (probabilidade mais baixa).

In [ ]:
n_show = min(5, len(fn_sorted))
if n_show > 0:
    fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3))
    if n_show == 1:
        axes = [axes]
    for j, idx in enumerate(fn_sorted[:n_show]):
        uid = all_uids[idx]
        xyz = all_xyzs[idx]
        ct = get_ct(uid)
        crop, _ = ct.extract_crop(xyz)
        axes[j].imshow(crop[16], cmap="gray")
        axes[j].set_title(f"prob={probs_a[idx]:.3f}")
        axes[j].axis("off")
        get_ct.cache_clear()
    plt.suptitle("Falsos negativos (nodulos perdidos)")
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum falso negativo com threshold=0.5")

Falsos positivos: alarmes falsos
Agora os candidatos que o modelo marcou como nodulo, mas que na verdade nao sao. Vamos ver os que tiveram maior confianca -- os alarmes mais "convictos".

In [ ]:
n_show = min(5, len(fp_sorted))
if n_show > 0:
    fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3))
    if n_show == 1:
        axes = [axes]
    for j, idx in enumerate(fp_sorted[:n_show]):
        uid = all_uids[idx]
        xyz = all_xyzs[idx]
        ct = get_ct(uid)
        crop, _ = ct.extract_crop(xyz)
        axes[j].imshow(crop[16], cmap="gray")
        axes[j].set_title(f"prob={probs_a[idx]:.3f}")
        axes[j].axis("off")
        get_ct.cache_clear()
    plt.suptitle("Falsos positivos (alarmes falsos)")
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum falso positivo com threshold=0.5")

Olhando nos tres planos
A fatia axial (de cima pra baixo) e a mais comum, mas nem sempre conta a historia toda. Vamos pegar um caso de cada tipo e ver nos tres planos anatomicos: axial, coronal (de frente) e sagital (de lado). E o mesmo que fizemos no notebook 04 quando exploramos os CTs.

In [ ]:
def show_three_planes(crop, title):
    d, h, w = crop.shape
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(crop[d // 2], cmap="gray")
    axes[0].set_title("Axial")
    axes[1].imshow(crop[:, h // 2, :], cmap="gray")
    axes[1].set_title("Coronal")
    axes[2].imshow(crop[:, :, w // 2], cmap="gray")
    axes[2].set_title("Sagital")
    for ax in axes:
        ax.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
if len(fn_sorted) > 0:
    idx = fn_sorted[0]
    ct = get_ct(all_uids[idx])
    crop, _ = ct.extract_crop(all_xyzs[idx])
    show_three_planes(crop, f"Falso negativo (prob={probs_a[idx]:.3f})")
    get_ct.cache_clear()

In [ ]:
if len(fp_sorted) > 0:
    idx = fp_sorted[0]
    ct = get_ct(all_uids[idx])
    crop, _ = ct.extract_crop(all_xyzs[idx])
    show_three_planes(crop, f"Falso positivo (prob={probs_a[idx]:.3f})")
    get_ct.cache_clear()

Os falsos positivos mais comuns costumam ser vasos sanguineos cortados transversalmente -- na fatia axial, eles parecem circulos pequenos, similares a nodulos. Os falsos negativos tipicamente sao nodulos muito pequenos ou que ficam muito proximos de estruturas anatomicas (parede do pulmao, mediastino), dificultando a distincao.

Distribuicao de probabilidades
Outro jeito de entender o modelo e olhar como as probabilidades se distribuem pra cada classe. Se o modelo for bom, esperamos ver duas "montanhas" separadas: nao-nodulos perto de 0 e nodulos perto de 1. A zona de sobreposicao e onde mora a incerteza -- e onde o threshold faz diferenca.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(probs_a[labels_a == 0], bins=50, alpha=0.6, density=True, label="Nao-nodulo")
plt.hist(probs_a[labels_a == 1], bins=50, alpha=0.6, density=True, label="Nodulo")
plt.xlabel("Probabilidade de nodulo")
plt.ylabel("Densidade")
plt.title("Distribuicao de probabilidades por classe")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Repare como a grande maioria dos nao-nodulos tem probabilidade proxima de zero, enquanto os nodulos tendem a ter probabilidade alta. A separacao e boa, confirmando o que a AUC ja indicava. A "cauda" dos nao-nodulos que se estende pra direita sao os falsos positivos -- e a cauda dos nodulos que fica a esquerda sao os falsos negativos.

Exportando o modulo de inferencia
Pra facilitar o uso no notebook de deploy (Gradio), a gente vai empacotar as funcoes de inferencia num modulo src/inference.py. Sao tres funcoes:

load_model(checkpoint_path) -- carrega o modelo e retorna junto com informacoes do treino
run_inference(candidates, model, ...) -- roda inferencia em uma lista de candidatos, agrupando por CT
classify_ct(series_uid, model, ...) -- classifica todos os candidatos de um CT especifico

In [ ]:
%%writefile ../src/inference.py
"""Funcoes de inferencia para o LunaModel."""

from collections import defaultdict

import numpy as np
import torch

from luna_data import load_candidates, get_ct
from model import LunaModel


def load_model(checkpoint_path, device=None):
    """Carrega o LunaModel a partir de um checkpoint.

    Retorna (model, info_dict) onde info_dict contem
    epoch, best_f1 e history.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model = LunaModel()
    model.load_state_dict(ckpt["model"])
    model.eval()
    model.to(device)
    info = {
        "epoch": ckpt["epoch"],
        "best_f1": ckpt["best_f1"],
        "history": ckpt.get("history"),
    }
    return model, info


def run_inference(candidates, model, device, batch_size=64,
                  max_cts=None, print_every=50):
    """Roda inferencia em uma lista de candidatos.

    Agrupa por series_uid pra carregar cada CT uma unica vez.
    Retorna dict com probs, labels, series_uids, center_xyzs.
    """
    uid_to_cands = defaultdict(list)
    for c in candidates:
        uid_to_cands[c.series_uid].append(c)

    uids = list(uid_to_cands.keys())
    if max_cts is not None:
        uids = uids[:max_cts]

    all_probs = []
    all_labels = []
    all_uids = []
    all_xyzs = []

    for i, uid in enumerate(uids):
        cands = uid_to_cands[uid]
        ct = get_ct(uid)

        crops = []
        labels = []
        xyzs = []
        for c in cands:
            crop_a, _ = ct.extract_crop(c.center_xyz)
            crops.append(crop_a)
            labels.append(int(c.is_nodule))
            xyzs.append(c.center_xyz)

        crops_t = torch.from_numpy(np.stack(crops)).float().unsqueeze(1)

        with torch.no_grad():
            for start in range(0, len(crops_t), batch_size):
                batch = crops_t[start:start + batch_size].to(device)
                _, probs = model(batch)
                all_probs.extend(probs[:, 1].cpu().numpy())

        all_labels.extend(labels)
        all_uids.extend([uid] * len(cands))
        all_xyzs.extend(xyzs)

        get_ct.cache_clear()

        if print_every > 0 and (i + 1) % print_every == 0:
            print(f"  {i + 1}/{len(uids)} CTs processados")

    return {
        "probs": np.array(all_probs),
        "labels": np.array(all_labels),
        "series_uids": all_uids,
        "center_xyzs": all_xyzs,
    }


def classify_ct(series_uid, model, device, batch_size=64):
    """Classifica todos os candidatos de um CT.

    Retorna lista de dicts ordenada por probabilidade (maior primeiro).
    Cada dict tem: series_uid, center_xyz, probability, is_nodule.
    """
    candidates = [
        c for c in load_candidates()
        if c.series_uid == series_uid
    ]
    if not candidates:
        return []

    ct = get_ct(series_uid)
    crops = []
    for c in candidates:
        crop_a, _ = ct.extract_crop(c.center_xyz)
        crops.append(crop_a)

    crops_t = torch.from_numpy(np.stack(crops)).float().unsqueeze(1)

    all_probs = []
    with torch.no_grad():
        for start in range(0, len(crops_t), batch_size):
            batch = crops_t[start:start + batch_size].to(device)
            _, probs = model(batch)
            all_probs.extend(probs[:, 1].cpu().numpy())

    get_ct.cache_clear()

    results = []
    for c, prob in zip(candidates, all_probs):
        results.append({
            "series_uid": c.series_uid,
            "center_xyz": c.center_xyz,
            "probability": float(prob),
            "is_nodule": c.is_nodule,
        })

    results.sort(key=lambda r: r["probability"], reverse=True)
    return results

In [ ]:
import importlib
import inference
importlib.reload(inference)

model_test, info = inference.load_model("../checkpoints/luna_model_best.pt", device)
print(f"Modelo carregado: epoca {info['epoch']}, F1={info['best_f1']:.4f}")

In [ ]:
test_uid = val_candidates[0].series_uid
results = inference.classify_ct(test_uid, model_test, device)

print(f"CT: {test_uid}")
print(f"Candidatos: {len(results)}")
print(f"\nTop 5 por probabilidade:")
for r in results[:5]:
    label = "nodulo" if r['is_nodule'] else "nao-nod"
    print(f"  prob={r['probability']:.4f}  [{label}]  xyz={r['center_xyz']}")

Conclusao
O modelo LunaModel treinado com 200 CTs mostra um desempenho solido pra um classificador de triagem:

Recall alto: detecta a grande maioria dos nodulos. Pra uso medico, isso e o mais importante -- nao queremos perder um cancer.
AUC alta: o modelo separa bem as classes, mesmo que a precisao bruta parecesse baixa por conta do desbalanceamento.
Erros interpretaveis: falsos positivos sao principalmente vasos sanguineos; falsos negativos sao nodulos atipicos ou muito pequenos.
No proximo notebook, a gente monta uma interface com Gradio pra demonstrar o modelo ao vivo: o usuario seleciona um CT, o modelo classifica todos os candidatos, e a interface mostra os resultados com visualizacoes.